## KNN Recommendation Model

In [1]:
# import
import numpy as np
import pandas as pd
import requests
from io import BytesIO
from PIL import Image

import torch
import torch.nn as nn

from sklearn.neighbors import NearestNeighbors
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

In [2]:
df = pd.read_csv('data/final_df.csv')
df.head()

,Unnamed: 0,filename,link,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_data
0,0,1638,http://assets.myntassets.com/v1/images/style/p...,1638,Unisex,Apparel,Bottomwear,Swimwear,Blue,Fall,2010.0,Sports,Nabaiji Swimming Goggles Blue Black,<PIL.Image.Image image mode=RGB size=1800x2400...
1,1,32903,http://assets.myntassets.com/v1/images/style/p...,32903,Women,Apparel,Bottomwear,Shorts,Red,Summer,2012.0,Casual,Allen Solly Women Red Shorts,<PIL.Image.Image image mode=RGB size=1080x1440...
2,2,39381,http://assets.myntassets.com/v1/images/style/p...,39381,Men,Apparel,Bottomwear,Jeans,Black,Summer,2012.0,Casual,Peter England Men Party Black Jeans,<PIL.Image.Image image mode=RGB size=1800x2400...
3,3,12163,http://assets.myntassets.com/v1/images/style/p...,12163,Women,Apparel,Bottomwear,Leggings,Purple,Fall,2011.0,Ethnic,Aurelia Women Solid Purple Leggings,<PIL.Image.Image image mode=RGB size=1800x2400...
4,4,1607,http://assets.myntassets.com/v1/images/style/p...,1607,Men,Apparel,Bottomwear,Track Pants,Blue,Fall,2010.0,Sports,Reebok Men trackpant- male Track Pants,<PIL.Image.Image image mode=RGB size=1080x1440...


In [3]:
# dataframe
print(df.columns)
print(df.shape)

Index(['Unnamed: 0', 'filename', 'link', 'id', 'gender', 'masterCategory',
       'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage',
       'productDisplayName', 'image_data'],
      dtype='object')
(1500, 14)


In [4]:
# image preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])])

In [5]:
# load pretrained ResNet-18 model and remove the final classification layer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

weights = ResNet18_Weights.DEFAULT
base_model = resnet18(weights=weights)

feature_extractor = nn.Sequential(*list(base_model.children())[:-1])
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/tiandrathreat/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:02<00:00, 17.8MB/s]


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [ ]:
# feature extraction function
def get_image_feature(url):
    try:
        response = requests.get(url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert("RGB")
        img_tensor = transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            feature = feature_extractor(img_tensor)

        return feature.cpu().numpy().flatten()

    except Exception as e:
        print(f"Error loading image: {url}")
        return None